# Bölüm 4.6: GPT Model - Detaylı İnceleme

Bu notebook, Bölüm 4.6'daki GPT Model mimarisini tüm detaylarıyla incelemektedir.

---

## 1. GPTModel Sınıfı - Bütün Bileşenler

### 1.1 Sınıf Tanımı

In [3]:
import torch
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        
        self.ff = nn.Sequential(
            nn.Linear(cfg["emb_dim"], cfg["emb_dim"] * 4),
            nn.GELU(),
            nn.Linear(cfg["emb_dim"] * 4, cfg["emb_dim"]))
        
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # === 1. Token Embedding ===
        self.tok_emb = nn.Embedding(
            num_embeddings=cfg["vocab_size"], 
            embedding_dim=cfg["emb_dim"]
        )
        
        # === 2. Pozisyon Embedding ===
        self.pos_emb = nn.Embedding(
            num_embeddings=cfg["context_length"], 
            embedding_dim=cfg["emb_dim"]
        )
        
        # === 3. Dropout ===
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # === 4. Transformer Block'lar ===
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        
        # === 5. Final LayerNorm ===
        self.final_norm = LayerNorm(cfg["emb_dim"])
        
        # === 6. Output Head ===
        self.out_head = nn.Linear(
            cfg["emb_dim"], 
            cfg["vocab_size"], 
            bias=False  # Weight tying için bias yok!
        )

    def forward(self, in_idx):
        # Input: [batch_size, seq_len]
        batch_size, seq_len = in_idx.shape
        
        # Token embedding'ler
        tok_embeds = self.tok_emb(in_idx)  # [batch, seq, emb_dim]
        
        # Pozisyon embedding'ler
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        
        # Embedding'leri topla
        x = tok_embeds + pos_embeds  # [batch, seq, emb_dim]
        
        # Dropout uygula
        x = self.drop_emb(x)
        
        # Transformer block'lardan geçir
        x = self.trf_blocks(x)
        
        # Final norm
        x = self.final_norm(x)
        
        # Output logits
        logits = self.out_head(x)  # [batch, seq, vocab_size]
        
        return logits

---

## 2. GPT-2 Konfigürasyonları

In [4]:
# GPT-2 124M (en küçük versiyon)
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Token sayısı
    "context_length": 1024,  # Maksimum sequence uzunluğu
    "emb_dim": 768,          # Embedding boyutu
    "n_heads": 12,           # Attention head sayısı
    "n_layers": 12,          # Transformer block sayısı
    "drop_rate": 0.1,        # Dropout oranı
    "qkv_bias": False        # QKV bias kullanılmasın
}

# Diğer GPT-2 versiyonları
GPT_CONFIG_355M = {  # medium
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.1,
    "qkv_bias": False
}

GPT_CONFIG_774M = {  # large
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1280,
    "n_heads": 20,
    "n_layers": 36,
    "drop_rate": 0.1,
    "qkv_bias": False
}

GPT_CONFIG_1558M = {  # XL
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1600,
    "n_heads": 25,
    "n_layers": 48,
    "drop_rate": 0.1,
    "qkv_bias": False
}

---

## 3. Model Oluşturma ve Parametre Analizi

In [5]:
# Modeli oluştur
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

# Model yapısını yazdır
print("=== GPT Model Yapısı ===\n")
print(model)

=== GPT Model Yapısı ===

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): Sequential(
        (0): Linear(in_features=768, out_features=3072, bias=True)
        (1): GELU(approximate='none')
        (2): Linear(in_features=3072, out_features=768, bias=True)
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=7

---

## 4. Parametre Hesaplama Fonksiyonu

In [6]:
def print_model_parameters(model, verbose=False):
    """Model parametrelerini detaylı olarak hesapla"""
    
    param_dict = {}
    total_params = 0
    
    for name, param in model.named_parameters():
        num_params = param.numel()
        param_dict[name] = num_params
        total_params += num_params
        
        if verbose and "weight" in name:
            print(f"{name}: {list(param.shape)} = {num_params:,} params")
    
    print(f"\n{'='*50}")
    print(f"TOPLAM PARAMETRE SAYISI: {total_params:,}")
    print(f"{'='*50}")
    
    return param_dict, total_params


# Parametreleri yazdır
param_dict, total_params = print_model_parameters(model, verbose=True)

tok_emb.weight: [50257, 768] = 38,597,376 params
pos_emb.weight: [1024, 768] = 786,432 params
trf_blocks.0.att.W_query.weight: [768, 768] = 589,824 params
trf_blocks.0.att.W_key.weight: [768, 768] = 589,824 params
trf_blocks.0.att.W_value.weight: [768, 768] = 589,824 params
trf_blocks.0.att.out_proj.weight: [768, 768] = 589,824 params
trf_blocks.0.ff.0.weight: [3072, 768] = 2,359,296 params
trf_blocks.0.ff.2.weight: [768, 3072] = 2,359,296 params
trf_blocks.1.att.W_query.weight: [768, 768] = 589,824 params
trf_blocks.1.att.W_key.weight: [768, 768] = 589,824 params
trf_blocks.1.att.W_value.weight: [768, 768] = 589,824 params
trf_blocks.1.att.out_proj.weight: [768, 768] = 589,824 params
trf_blocks.1.ff.0.weight: [3072, 768] = 2,359,296 params
trf_blocks.1.ff.2.weight: [768, 3072] = 2,359,296 params
trf_blocks.2.att.W_query.weight: [768, 768] = 589,824 params
trf_blocks.2.att.W_key.weight: [768, 768] = 589,824 params
trf_blocks.2.att.W_value.weight: [768, 768] = 589,824 params
trf_blocks.

---

## 5. Her Katmanın Parametre Sayısı

In [7]:
import pandas as pd

def calculate_layer_parameters():
    """Her katman için parametre hesapla"""
    
    vocab_size = 50257
    context_length = 1024
    emb_dim = 768
    n_heads = 12
    n_layers = 12
    
    # 1. Token Embedding
    tok_emb_params = vocab_size * emb_dim
    
    # 2. Positional Embedding
    pos_emb_params = context_length * emb_dim
    
    # 3. Her Transformer Block
    # Attention
    qkv_params = 3 * (emb_dim * emb_dim)  # W_query, W_key, W_value
    out_proj_params = emb_dim * emb_dim
    
    # Feed Forward
    ffn_up = emb_dim * (emb_dim * 4)      # 768 -> 3072
    ffn_down = (emb_dim * 4) * emb_dim   # 3072 -> 768
    
    # LayerNorm (scale + shift)
    ln_params = 2 * emb_dim
    
    # Dropout - parametresiz!
    
    attention_params = qkv_params + out_proj_params
    ffn_params = ffn_up + ffn_down
    block_params = attention_params + ffn_params + ln_params
    
    # 4. Final LayerNorm
    final_ln_params = 2 * emb_dim
    
    # 5. Output Head
    out_head_params = emb_dim * vocab_size
    
    # Tablo oluştur
    data = {
        "Katman": [
            "Token Embedding",
            "Positional Embedding", 
            "1 Transformer Block",
            "  - MultiHeadAttention",
            "    - W_query + W_key + W_value",
            "    - out_proj",
            "  - FeedForward",
            "    - Linear Up (768->3072)",
            "    - Linear Down (3072->768)",
            "  - LayerNorm × 2",
            f"Transformer Blocks × {n_layers}",
            "Final LayerNorm",
            "Output Head",
            "TOPLAM (Weight Tying yok)",
            "TOPLAM (Weight Tying var)"
        ],
        "Formül": [
            f"{vocab_size:,} × {emb_dim}",
            f"{context_length:,} × {emb_dim}",
            "-",
            "-",
            f"3 × ({emb_dim} × {emb_dim})",
            f"{emb_dim} × {emb_dim}",
            "-",
            f"{emb_dim} × {emb_dim*4}",
            f"{emb_dim*4} × {emb_dim}",
            f"2 × {emb_dim}",
            f"× {n_layers}",
            f"2 × {emb_dim}",
            f"{emb_dim} × {vocab_size:,}",
            "-",
            "-"
        ],
        "Parametre": [
            f"{tok_emb_params:,}",
            f"{pos_emb_params:,}",
            f"{block_params:,}",
            f"{attention_params:,}",
            f"{qkv_params:,}",
            f"{out_proj_params:,}",
            f"{ffn_params:,}",
            f"{ffn_up:,}",
            f"{ffn_down:,}",
            f"{ln_params:,}",
            f"{block_params * n_layers:,}",
            f"{final_ln_params:,}",
            f"{out_head_params:,}",
            f"{tok_emb_params + pos_emb_params + block_params * n_layers + final_ln_params + out_head_params:,}",
            f"{tok_emb_params + pos_emb_params + block_params * n_layers + final_ln_params:,}"
        ]
    }
    
    df = pd.DataFrame(data)
    return df

# Tabloyu yazdır
df = calculate_layer_parameters()
print(df.to_string(index=False))

                         Katman          Formül   Parametre
                Token Embedding    50,257 × 768  38,597,376
           Positional Embedding     1,024 × 768     786,432
            1 Transformer Block               -   7,079,424
           - MultiHeadAttention               -   2,359,296
    - W_query + W_key + W_value 3 × (768 × 768)   1,769,472
                     - out_proj       768 × 768     589,824
                  - FeedForward               -   4,718,592
        - Linear Up (768->3072)      768 × 3072   2,359,296
      - Linear Down (3072->768)      3072 × 768   2,359,296
                - LayerNorm × 2         2 × 768       1,536
        Transformer Blocks × 12            × 12  84,953,088
                Final LayerNorm         2 × 768       1,536
                    Output Head    768 × 50,257  38,597,376
      TOPLAM (Weight Tying yok)               - 162,935,808
      TOPLAM (Weight Tying var)               - 124,338,432


---

## 6. Weight Tying Açıklaması

In [8]:
print("=" * 60)
print("WEIGHT TYING NEDİR?")
print("=" * 60)

# Modeli oluştur
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

print("\n1. Weight Tying ÖNCESİ:")
print(f"   Token Embedding: {model.tok_emb.weight.shape}")
print(f"   Output Head:     {model.out_head.weight.shape}")

# Token embedding ile output head'i eşitle
model.out_head.weight = model.tok_emb.weight

print("\n2. Weight Tying SONRASI:")
print(f"   Token Embedding: {model.tok_emb.weight.shape}")
print(f"   Output Head:     {model.out_head.weight.shape}")
print(f"   AYNI MEMORY ADRESİ: {model.tok_emb.weight.data_ptr() == model.out_head.weight.data_ptr()}")

# Yeni parametre sayısı
total_with_tying = sum(p.numel() for p in model.parameters())
print(f"\n3. PARAMETRE KARŞILAŞTIRMASI:")
print(f"   Weight Tying yok: 163,009,536")
print(f"   Weight Tying var: {total_with_tying:,}")
print(f"   Tasarruf:         {163009536 - total_with_tying:,}")

WEIGHT TYING NEDİR?

1. Weight Tying ÖNCESİ:
   Token Embedding: torch.Size([50257, 768])
   Output Head:     torch.Size([50257, 768])

2. Weight Tying SONRASI:
   Token Embedding: torch.Size([50257, 768])
   Output Head:     torch.Size([50257, 768])
   AYNI MEMORY ADRESİ: True

3. PARAMETRE KARŞILAŞTIRMASI:
   Weight Tying yok: 163,009,536
   Weight Tying var: 124,412,160
   Tasarruf:         38,597,376


### Weight Tying Görselleştirme

```
Weight Tying ÖNCESİ:
┌─────────────────────┐      ┌─────────────────────┐
│   tok_emb.weight    │      │   out_head.weight   │
│  [50257, 768]       │      │  [50257, 768]       │
│   38,597,376 params │      │   38,597,376 params │
└─────────────────────┘      └─────────────────────┘


Weight Tying SONRASI:
┌─────────────────────┐      
│   tok_emb.weight    │──────┐
│  [50257, 768]       │      │ AYNı
│   38,597,376 params │<─────┘
└─────────────────────┘      
    (out_head bu reference'ı kullanır)
```

---

## 7. Forward Pass Detaylı İnceleme

In [9]:
import matplotlib.pyplot as plt
import numpy as np

def visualize_forward_pass():
    """Forward pass'i adım adım görselleştir"""
    
    # Basit bir örnek
    torch.manual_seed(123)
    model = GPTModel(GPT_CONFIG_124M)
    model.eval()
    
    # Input: "Hello world" (basit token ID'ler)
    input_ids = torch.tensor([[1, 2, 3, 4, 5]])  # [batch=1, seq=5]
    
    print("=" * 60)
    print("FORWARD PASS ADIMLARI")
    print("=" * 60)
    
    with torch.no_grad():
        # Step 1: Token Embedding
        tok_embeds = model.tok_emb(input_ids)
        print(f"\n1. Token Embedding:")
        print(f"   Input shape:  {input_ids.shape}")
        print(f"   Output shape: {tok_embeds.shape}")
        print(f"   Değerler: min={tok_embeds.min():.4f}, max={tok_embeds.max():.4f}")
        
        # Step 2: Positional Embedding
        seq_len = input_ids.shape[1]
        pos_embeds = model.pos_emb(torch.arange(seq_len))
        print(f"\n2. Positional Embedding:")
        print(f"   Output shape: {pos_embeds.shape}")
        
        # Step 3: Embedding toplama
        x = tok_embeds + pos_embeds
        print(f"\n3. Embedding Toplamı:")
        print(f"   Output shape: {x.shape}")
        
        # Step 4: Transformer Blocks
        x = model.trf_blocks(x)
        print(f"\n4. Transformer Blocks (×12):")
        print(f"   Output shape: {x.shape}")
        
        # Step 5: Final LayerNorm
        x = model.final_norm(x)
        print(f"\n5. Final LayerNorm:")
        print(f"   Output shape: {x.shape}")
        
        # Step 6: Output Head
        logits = model.out_head(x)
        print(f"\n6. Output Head:")
        print(f"   Output shape: {logits.shape}")
        print(f"   (Her pozisyon için {logits.shape[-1]} token olasılığı)")
        
    return logits

# Forward pass'i çalıştır
logits = visualize_forward_pass()

FORWARD PASS ADIMLARI

1. Token Embedding:
   Input shape:  torch.Size([1, 5])
   Output shape: torch.Size([1, 5, 768])
   Değerler: min=-3.9299, max=3.6863

2. Positional Embedding:
   Output shape: torch.Size([5, 768])

3. Embedding Toplamı:
   Output shape: torch.Size([1, 5, 768])

4. Transformer Blocks (×12):
   Output shape: torch.Size([1, 5, 768])

5. Final LayerNorm:
   Output shape: torch.Size([1, 5, 768])

6. Output Head:
   Output shape: torch.Size([1, 5, 50257])
   (Her pozisyon için 50257 token olasılığı)


---

## 8. Memory Hesaplama

In [10]:
def calculate_memory(model, precision="float32"):
    """Model memory gereksinimini hesapla"""
    
    total_params = sum(p.numel() for p in model.parameters())
    
    # Precision'a göre byte hesapla
    if precision == "float32":
        bytes_per_param = 4
    elif precision == "float16":
        bytes_per_param = 2
    elif precision == "bfloat16":
        bytes_per_param = 2
    elif precision == "int8":
        bytes_per_param = 1
    else:
        bytes_per_param = 4
    
    # Store gradients (training için)
    grad_params = total_params  # Aynı boyutta
    
    # Optimizer state (AdamW için ~2x)
    optimizer_params = total_params * 2
    
    total_bytes = (total_params + grad_params + optimizer_params) * bytes_per_param
    total_mb = total_bytes / (1024 * 1024)
    total_gb = total_mb / 1024
    
    print(f"=" * 60)
    print(f"MEMORY HESAPLAMA ({precision})")
    print(f"=" * 60)
    print(f"Model parametreleri:  {total_params:>15,} ({total_params*bytes_per_param/(1024**2):.2f} MB)")
    print(f"Gradients:            {grad_params:>15,} ({grad_params*bytes_per_param/(1024**2):.2f} MB)")
    print(f"Optimizer state:      {optimizer_params:>15,} ({optimizer_params*bytes_per_param/(1024**2):.2f} MB)")
    print(f"{'=' * 60}")
    print(f"TOPLAM:               {total_params + grad_params + optimizer_params:>15,} ({total_mb:.2f} MB = {total_gb:.2f} GB)")
    
    return total_mb

# Memory hesapla
memory = calculate_memory(model, "float32")

MEMORY HESAPLAMA (float32)
Model parametreleri:      124,412,160 (474.59 MB)
Gradients:                124,412,160 (474.59 MB)
Optimizer state:          248,824,320 (949.19 MB)
TOPLAM:                   497,648,640 (1898.38 MB = 1.85 GB)


---

## 9. Farklı GPT-2 Versiyonlarının Karşılaştırması

In [11]:
def compare_gpt2_versions():
    """Tüm GPT-2 versiyonlarını karşılaştır"""
    
    versions = {
        "GPT2-small (124M)": {
            "emb_dim": 768, "n_layers": 12, "n_heads": 12, "vocab_size": 50257
        },
        "GPT2-medium (355M)": {
            "emb_dim": 1024, "n_layers": 24, "n_heads": 16, "vocab_size": 50257
        },
        "GPT2-large (774M)": {
            "emb_dim": 1280, "n_layers": 36, "n_heads": 20, "vocab_size": 50257
        },
        "GPT2-XL (1.5B)": {
            "emb_dim": 1600, "n_layers": 48, "n_heads": 25, "vocab_size": 50257
        }
    }
    
    results = []
    
    for name, cfg in versions.items():
        vocab = cfg["vocab_size"]
        emb = cfg["emb_dim"]
        layers = cfg["n_layers"]
        
        # Basit parametre hesabı
        embed = vocab * emb
        pos = 1024 * emb
        
        # Transformer block (yaklaşık)
        att = 3 * emb * emb + emb * emb
        ffn = 2 * (emb * emb * 4)
        block = att + ffn + 2 * emb
        
        total = embed + pos + block * layers + 2 * emb + vocab * emb
        
        results.append({
            "Versiyon": name,
            "emb_dim": emb,
            "Layers": layers,
            "Heads": cfg["n_heads"],
            "Parametre (M)": total / 1_000_000,
            "Memory (FP32, GB)": total * 4 / (1024**3)
        })
    
    return pd.DataFrame(results)

# Karşılaştırma tablosu
df_versions = compare_gpt2_versions()
print(df_versions.to_string(index=False))

          Versiyon  emb_dim  Layers  Heads  Parametre (M)  Memory (FP32, GB)
 GPT2-small (124M)      768      12     12     162.935808           0.606983
GPT2-medium (355M)     1024      24     16     406.016000           1.512527
 GPT2-large (774M)     1280      36     20     837.852160           3.121243
    GPT2-XL (1.5B)     1600      48     25    1637.177600           6.098962


---

## 10. Model Yapısı Görselleştirme

In [12]:
def draw_gpt_architecture():
    """GPT mimarisi ASCII diyagramı"""
    
    diagram = """
╔══════════════════════════════════════════════════════════════════════╗
║                         GPT MODEL MİMARİSİ                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Input: "The cat sat"                                                ║
║   ↓                                                                  ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 1. TOKEN EMBEDDING LAYER                                      │   ║
║  │    nn.Embedding(50257, 768)                                  │   ║
║  │    Input:  [batch, seq]  →  [batch, seq, 768]                │   ║
║  │    Param:  38,597,376                                         │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓                                                                  ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 2. POSITIONAL EMBEDDING                                       │   ║
║  │    nn.Embedding(1024, 768)                                   │   ║
║  │    Input:  [seq]  →  [seq, 768]                              │   ║
║  │    Param:  786,432                                            │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓  (toplama)                                                      ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 3. DROPOUT (10%)                                             │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓                                                                  ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 4. TRANSFORMER BLOCK × 12                                    │   ║
║  │    ┌────────────────────────────────────────────────────┐    │   ║
║  │    │ Block i:                                            │    │   ║
║  │    │  ├── LayerNorm                                      │    │   ║
║  │    │  ├── MultiHeadAttention → + (shortcut)              │    │   ║
║  │    │  ├── LayerNorm                                      │    │   ║
║  │    │  ├── FeedForward (GELU) → + (shortcut)              │    │   ║
║  │    └────────────────────────────────────────────────────┘    │   ║
║  │    Param/Block: ~7,079,424                                    │   ║
║  │    × 12 = 84,953,088                                         │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓                                                                  ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 5. FINAL LAYERNORM                                            │   ║
║  │    Param: 1,536                                                │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓                                                                  ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 6. OUTPUT HEAD                                                 │   ║
║  │    nn.Linear(768, 50257, bias=False)                         │   ║
║  │    Input:  [batch, seq, 768]  →  [batch, seq, 50257]         │   ║
║  │    Param:  38,597,376                                         │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓                                                                  ║
║  Output: [batch, seq, 50257]  →  Softmax → Token ID'ler            ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""
    print(diagram)

draw_gpt_architecture()


╔══════════════════════════════════════════════════════════════════════╗
║                         GPT MODEL MİMARİSİ                          ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  Input: "The cat sat"                                                ║
║   ↓                                                                  ║
║  ┌────────────────────────────────────────────────────────────────┐   ║
║  │ 1. TOKEN EMBEDDING LAYER                                      │   ║
║  │    nn.Embedding(50257, 768)                                  │   ║
║  │    Input:  [batch, seq]  →  [batch, seq, 768]                │   ║
║  │    Param:  38,597,376                                         │   ║
║  └────────────────────────────────────────────────────────────────┘   ║
║   ↓                                                                  ║
║  ┌───────────────────────────────────────────────

---

## 11. Özet

In [13]:
def print_summary():
    summary = """
    ╔═══════════════════════════════════════════════════════════════════╗
    ║                    BÖLÜM 4.6 ÖZETİ                                ║
    ╠═══════════════════════════════════════════════════════════════════╣
    ║                                                                   ║
    ║  ✅ GPTModel Sınıfı:                                              ║
    ║     - Token Embedding (vocab_size × emb_dim)                     ║
    ║     - Positional Embedding (context_length × emb_dim)            ║
    ║     - Transformer Block × n_layers                                ║
    ║     - Final LayerNorm                                            ║
    ║     - Output Head (emb_dim × vocab_size)                          ║
    ║                                                                   ║
    ║  ✅ Parametre Hesabı (GPT2-124M):                                 ║
    ║     - Weight Tying yok: 163,009,536                              ║
    ║     - Weight Tying var:  124,412,160                             ║
    ║                                                                   ║
    ║  ✅ Memory Gereksinimi (FP32):                                    ║
    ║     - Sadece model: ~621 MB                                      ║
    ║     - + Gradients: ~621 MB                                       ║
    ║     - + Optimizer:  ~1.2 GB                                      ║
    ║     - Toplam:      ~2.5 GB                                        ║
    ║                                                                   ║
    ║  ✅ Forward Pass Çıktıs:                                          ║
    ║     - Shape: [batch, seq_len, vocab_size=50257]                 ║
    ║     - Her pozisyon için tüm token olasılıkları                   ║
    ║                                                                   ║
    ╚═══════════════════════════════════════════════════════════════════╝
    """
    print(summary)

print_summary()


    ╔═══════════════════════════════════════════════════════════════════╗
    ║                    BÖLÜM 4.6 ÖZETİ                                ║
    ╠═══════════════════════════════════════════════════════════════════╣
    ║                                                                   ║
    ║  ✅ GPTModel Sınıfı:                                              ║
    ║     - Token Embedding (vocab_size × emb_dim)                     ║
    ║     - Positional Embedding (context_length × emb_dim)            ║
    ║     - Transformer Block × n_layers                                ║
    ║     - Final LayerNorm                                            ║
    ║     - Output Head (emb_dim × vocab_size)                          ║
    ║                                                                   ║
    ║  ✅ Parametre Hesabı (GPT2-124M):                                 ║
    ║     - Weight Tying yok: 163,009,536                              ║
    ║     - Weight Tying var:  124,412,160 

---

## Referans

Bu notebook, Sebastian Raschka'nın "Build a Large Language Model From Scratch" 
kitabının Bölüm 4.6'sından hazırlanmıştır.

Daha fazla bilgi için:
- `ch04.ipynb` - Ana not defteri
- `BOLUM4_OZETI.md` - Bölüm özeti